In [ ]:
#| default_exp size

In [ ]:
#| include: false
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations

from fasterbench.core import _bytes_to_mib, _is_quantized
import torch
import io
from dataclasses import dataclass, asdict

In [ ]:
#| export
def get_model_size(model: torch.nn.Module) -> int:  # model to measure
    """Return the on-disk size (bytes) of the serialized model."""
    buf = io.BytesIO()
    try:
        model.save(buf)
    except Exception:
        torch.save(model.state_dict(), buf)
    return buf.getbuffer().nbytes


#| export
def _quantized_param_count(model: torch.nn.Module) -> int:  # model with quantized modules
    """Sum the numel of packed quantized weights/biases (not in `.parameters()`).

    Quantized `Conv2d`/`Linear` keep their weights in `_packed_params`, exposed via
    a *callable* `.weight()`/`.bias()` accessor (unlike a plain `nn.Parameter`, which
    is a tensor). Quantization preserves the number of parameters, so counting these
    makes a quantized model's param count match its fp32 twin.
    """
    total = 0
    for m in model.modules():
        for name in ("weight", "bias"):
            accessor = getattr(m, name, None)
            if not callable(accessor):  # plain nn.Parameter tensors are not callable
                continue
            try:
                t = accessor()
                if t is not None:
                    total += t.numel()
            except Exception:
                pass
    return total


#| export
def get_num_parameters(
    model: torch.nn.Module,       # model to count parameters
    trainable_only: bool = True,  # if True, only count trainable parameters
) -> int:
    """Count the number of (optionally trainable) parameters.

    Quantized weights live in packed params outside `.parameters()`, so they are
    added separately (regardless of `trainable_only`, as they are never trainable).
    """
    if trainable_only:
        n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    else:
        n = sum(p.numel() for p in model.parameters())
    if _is_quantized(model):
        n += _quantized_param_count(model)
    return n


#| export
@dataclass(slots=True)
class SizeMetrics:
    """Model size metrics: disk size and parameter count."""
    disk_bytes: int
    size_mib: float
    num_params: int

    def as_dict(self) -> dict[str, float]:
        return asdict(self)


#| export
def compute_size(
    model: torch.nn.Module,             # model to measure
    *,
    params_count: int | None = None,    # pre-computed parameter count (avoids recount)
) -> SizeMetrics:
    """Compute size metrics for a model."""
    disk = get_model_size(model)
    params = params_count if params_count is not None else get_num_parameters(model)
    return SizeMetrics(disk_bytes=disk, size_mib=_bytes_to_mib(disk), num_params=params)

In [ ]:
show_doc(SizeMetrics)

In [ ]:
show_doc(compute_size)

In [ ]:
show_doc(get_model_size)

In [ ]:
show_doc(get_num_parameters)

In [ ]:
#| hide
from fastcore.test import *

import torch.nn as nn
_m = nn.Linear(10, 5)
_s = compute_size(_m)
assert isinstance(_s, SizeMetrics)
assert _s.num_params > 0
test_eq(get_num_parameters(_m), 55)

In [ ]:
#| hide
# Quantization preserves the *number* of params (int8 conv/linear have the same
# weight count as their fp32 twin) — only the bytes shrink. But quantized weights
# live in packed params outside `.parameters()`, so the naive sum reports 0.
# Verify: quantized param count is non-zero and ~equal to the fp32 count.
import torch.nn as nn
from torch.ao.quantization import get_default_qconfig, QConfigMapping
from torch.ao.quantization.quantize_fx import prepare_fx, convert_fx
from fasterbench.core import _is_quantized

class _SizeNet(nn.Module):
    "Small conv+linear net, statically quantizable via FX."
    def __init__(self):
        super().__init__()
        self.c1, self.c2 = nn.Conv2d(3, 16, 3, padding=1), nn.Conv2d(16, 32, 3, padding=1)
        self.pool, self.flat, self.fc = nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(32, 10)
    def forward(self, x):
        x = torch.relu(self.c1(x)); x = torch.relu(self.c2(x))
        return self.fc(self.flat(self.pool(x)))

_fp = _SizeNet().eval()
_x = torch.randn(1, 3, 16, 16)
_fp_params = get_num_parameters(_fp)                # fp32 baseline

torch.backends.quantized.engine = "x86"
_qmap = QConfigMapping().set_global(get_default_qconfig("x86"))
_prep = prepare_fx(_SizeNet().eval(), _qmap, example_inputs=(_x,))
_prep(_x)                                           # calibrate
_qm = convert_fx(_prep)
assert _is_quantized(_qm)

# BEFORE the fix, the naive `.parameters()` sum is 0 for a fully quantized model.
test_eq(sum(p.numel() for p in _qm.parameters() if p.requires_grad), 0)

_q_params = get_num_parameters(_qm)                 # AFTER the fix
assert _q_params > 0                                # zero is gone
assert abs(_q_params - _fp_params) <= 0.05 * _fp_params   # within ~5% of fp32
# trainable_only must NOT hide quantized weights (they are never trainable)
test_eq(get_num_parameters(_qm, trainable_only=False), _q_params)

---

## See Also

- [Benchmark](../analysis/benchmark.html) - Unified benchmarking with `benchmark()`
- [Compute](compute.html) - MACs and operation counts
- [Profiling](../analysis/profiling.html) - Per-layer size analysis